In [2]:
# ! uv pip install scikit-learn
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline


In [3]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "Data"
    / "processed"
    / "nyc_311_regression.csv"
)

df = pd.read_csv(DATA_PATH, parse_dates=["created_date"])

In [4]:
print("Dataset shape:", df.shape)

Dataset shape: (4668, 38)


In [5]:
df.head()

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,...,y_coordinate_state_plane,open_data_channel_type,park_facility_name,park_borough,latitude,longitude,location,is_closed,resolution_hours,completion_hours
0,71179718,2026-11-19 01:42:26,2026-11-19 02:04:36,NYPD,NEW YORK CITY POLICE DEPARTMENT,NOISE - RESIDENTIAL,BANGING/POUNDING,NaN,RESIDENTIAL BUILDING/HOUSE,10039.0,...,242527.0,PHONE,Unspecified,MANHATTAN,40.832338,-73.935432,POINT (-73.935431738007 40.832337915244),1,0.369444,0.369444
1,71171016,2026-11-19 01:34:18,2026-11-19 01:47:10,NYPD,NEW YORK CITY POLICE DEPARTMENT,NOISE - RESIDENTIAL,LOUD TALKING,NaN,RESIDENTIAL BUILDING/HOUSE,10009.0,...,206421.0,ONLINE,Unspecified,MANHATTAN,40.733252,-73.976030,POINT (-73.976030383539 40.733252130601),1,0.214444,0.214444
2,71171039,2026-11-19 01:33:55,2026-11-19 01:41:02,NYPD,NEW YORK CITY POLICE DEPARTMENT,NOISE - STREET/SIDEWALK,LOUD TALKING,NaN,STREET/SIDEWALK,10024.0,...,226231.0,MOBILE,Unspecified,MANHATTAN,40.787626,-73.978936,POINT (-73.978935878994 40.787626067455),1,0.118611,0.118611
3,71177582,2026-11-19 01:28:48,2026-11-19 01:54:06,NYPD,NEW YORK CITY POLICE DEPARTMENT,NOISE - COMMERCIAL,BANGING/POUNDING,NaN,CLUB/BAR/RESTAURANT,11211.0,...,199246.0,ONLINE,Unspecified,BROOKLYN,40.713555,-73.961886,POINT (-73.961886477712 40.713554678518),1,0.421667,0.421667
4,71174389,2026-11-19 01:27:53,2026-11-19 01:29:46,NYPD,NEW YORK CITY POLICE DEPARTMENT,NOISE - RESIDENTIAL,BANGING/POUNDING,NaN,RESIDENTIAL BUILDING/HOUSE,11214.0,...,160824.0,PHONE,Unspecified,BROOKLYN,40.608101,-73.999384,POINT (-73.999384145633 40.608101083647),1,0.031389,0.031389


### Feature engineering

In [6]:
candidate_features = [
    "created_date",
    "agency",
    "complaint_type",
    "descriptor",
    "location_type",
    "incident_zip",
    "address_type",
    "city",
    "community_board",
    "council_district",
    "police_precinct",
    "borough",
    "open_data_channel_type",
    "park_borough",
    "latitude",
    "longitude"
]

# Create feature dataset
model_df = df[candidate_features + ["completion_hours"]].copy()

In [7]:
model_df["created_hour"] = model_df["created_date"].dt.hour
model_df["created_dayofweek"] = model_df["created_date"].dt.dayofweek
model_df["created_month"] = model_df["created_date"].dt.month


C:\Users\SAUD\AppData\Local\Temp\ipykernel_15424\353169103.py:2: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  model_df["created_dayofweek"] = model_df["created_date"].dt.dayofweek


In [8]:
model_df = model_df.drop(columns=["created_date"])

In [9]:
X = model_df.drop(columns=["completion_hours"])
y = model_df["completion_hours"]

In [10]:
print("Feature matrix:", X.shape)
print("Target:", y.shape)

Feature matrix: (4668, 18)
Target: (4668,)


In [11]:
print("\nFeatures:")
print(X.columns.tolist())


Features:
['agency', 'complaint_type', 'descriptor', 'location_type', 'incident_zip', 'address_type', 'city', 'community_board', 'council_district', 'police_precinct', 'borough', 'open_data_channel_type', 'park_borough', 'latitude', 'longitude', 'created_hour', 'created_dayofweek', 'created_month']


In [12]:
print("\nTarget statistics:")
print(y.describe())


Target statistics:
count    4668.000000
mean        1.983038
std         2.491029
min         0.000000
25%         0.399931
50%         1.072361
75%         2.537778
max        18.416667
Name: completion_hours, dtype: float64


### Train/test split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (3734, 18)
Testing features: (934, 18)
Training target: (3734,)
Testing target: (934,)


In [14]:
from assignment_2.preprocessing import create_preprocessor

# Identify feature types
numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

# Create preprocessing pipeline
preprocessor = create_preprocessor(
    numeric_features,
    categorical_features
)

print("Preprocessor:", preprocessor)

Preprocessor: ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['incident_zip', 'council_district',
                                  'latitude', 'longitude', 'created_hour',
                                  'created_dayofweek', 'created_month']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['agency', 'complaint_type', 'descriptor',
                                  'location_type', 'address_type', 'city',
  

### baseline regression pipeline

In [15]:


baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DummyRegressor(strategy="median"))
])

print(baseline_model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['incident_zip',
                                                   'council_district',
                                                   'latitude', 'longitude',
                                                   'created_hour',
                                                   'created_dayofweek',
                                                   'created_month']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
       

### Evaluate the baseline using 5-fold cross-validation

In [16]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2"
}

baseline_scores = cross_validate(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

print("Baseline CV MAE:", -baseline_scores["test_mae"].mean())
print("Baseline CV RMSE:", -baseline_scores["test_rmse"].mean())
print("Baseline CV R2:", baseline_scores["test_r2"].mean())

Baseline CV MAE: 1.5226747816407125
Baseline CV RMSE: 2.6007903972168167
Baseline CV R2: -0.13773389488121776


### Random Forest model

In [17]:
from sklearn.ensemble import RandomForestRegressor

random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

print(random_forest_model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['incident_zip',
                                                   'council_district',
                                                   'latitude', 'longitude',
                                                   'created_hour',
                                                   'created_dayofweek',
                                                   'created_month']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
       

In [18]:
rf_scores = cross_validate(
    random_forest_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Random Forest CV MAE:", -rf_scores["test_mae"].mean())
print("Random Forest CV RMSE:", -rf_scores["test_rmse"].mean())
print("Random Forest CV R2:", rf_scores["test_r2"].mean())

Random Forest CV MAE: 0.9598447506229497
Random Forest CV RMSE: 1.6120906314238428
Random Forest CV R2: 0.5605474142209235


### Extra Trees model

In [19]:
from sklearn.ensemble import ExtraTreesRegressor

extra_trees_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", ExtraTreesRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

print(extra_trees_model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['incident_zip',
                                                   'council_district',
                                                   'latitude', 'longitude',
                                                   'created_hour',
                                                   'created_dayofweek',
                                                   'created_month']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
       

In [20]:
extra_trees_scores = cross_validate(
    extra_trees_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Extra Trees CV MAE:", -extra_trees_scores["test_mae"].mean())
print("Extra Trees CV RMSE:", -extra_trees_scores["test_rmse"].mean())
print("Extra Trees CV R2:", extra_trees_scores["test_r2"].mean())

Extra Trees CV MAE: 0.947594655197696
Extra Trees CV RMSE: 1.689214254393999
Extra Trees CV R2: 0.5168997503080935


In [21]:
comparison = pd.DataFrame({
    "Model": ["Median Baseline", "Random Forest", "Extra Trees"],
    "CV MAE": [
        -baseline_scores["test_mae"].mean(),
        -rf_scores["test_mae"].mean(),
        -extra_trees_scores["test_mae"].mean()
    ],
    "CV RMSE": [
        -baseline_scores["test_rmse"].mean(),
        -rf_scores["test_rmse"].mean(),
        -extra_trees_scores["test_rmse"].mean()
    ],
    "CV R2": [
        baseline_scores["test_r2"].mean(),
        rf_scores["test_r2"].mean(),
        extra_trees_scores["test_r2"].mean()
    ]
})

comparison

,Model,CV MAE,CV RMSE,CV R2
0,Median Baseline,1.522675,2.600790,-0.137734
1,Random Forest,0.959845,1.612091,0.560547
2,Extra Trees,0.947595,1.689214,0.516900


### training model

In [22]:
best_model = random_forest_model

best_model.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


### Save model

In [23]:
import joblib
from pathlib import Path

MODEL_DIR = PROJECT_ROOT / "outputs" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "nyc_311_completion_model.joblib"

joblib.dump(best_model, model_path)

print("Model saved at:", model_path)

Model saved at: d:\AI and AUTOMATION\assignments\assignment_2\outputs\models\nyc_311_completion_model.joblib
